# 08 — Paper tables, figures and RESULTS_SUMMARY.md

**Purpose.** Turn the saved results into paper-ready tables (CSV + Markdown + LaTeX) and figures (PNG), and list every number with its run ID, source file and label in `outputs/RESULTS_SUMMARY.md`.

**Kaggle settings.** Accelerator **None** · Internet Off is fine.

**Inputs.** Dataset `bdsl-k400` + **Add Input → Output of notebook 07**.

**Outputs** (under `/kaggle/working/k400/`, saved by *Save Version → Save & Run All*). `outputs/tables/` (T1–T8, per-item tables, RESULTS_LEDGER.csv), `outputs/figures/` (F1–F6), `outputs/RESULTS_SUMMARY.md`.

**Expected run time** [ESTIMATE, not measured yet]. under 3 minutes.

**What to check.** `final_test_done: true`; `missing` is empty; figures look right (curves, method bars with CIs, error analysis).

> Honesty rules: label every number measured or estimated; results are small-scale and signer-dependent;
> the test split is used once (notebook 07); low coordinate error is not proof of correct sign language;
> G0 glosses are pseudo-glosses, not BdSL annotation; no human or expert validation exists.

In [ ]:
# Cell 1 - Setup (the same in every notebook). Finds the data and the code, sets the paths, and copies the
# outputs of earlier notebooks (added with "Add Input") into /kaggle/working/k400 so results accumulate.
import os, sys, zipfile, pathlib, json
NOTEBOOK = "08_report_figures"

def _find_root():
    base = pathlib.Path("/kaggle/input")
    if base.is_dir():
        for dirpath, dirnames, filenames in os.walk(base):
            if "UPLOAD_MANIFEST.csv" in filenames and "PACKAGE_INFO.json" in filenames:
                return pathlib.Path(dirpath)
            depth = len(pathlib.Path(dirpath).relative_to(base).parts)
            dirnames[:] = [] if depth >= 4 else [d for d in dirnames if d not in ("videos", "k400")]
        zips = sorted(base.rglob("bdsl-k400-*.zip"))
        if zips:  # Kaggle kept the upload as one zip file: unpack it once (a few minutes)
            dest = pathlib.Path("/tmp/k400_dataset")   # /tmp is not saved as Output
            if not (dest / "UPLOAD_MANIFEST.csv").is_file():
                print("Unpacking", zips[0], "->", dest)
                zipfile.ZipFile(zips[0]).extractall(dest)
            return dest
        raise FileNotFoundError("The private dataset is not attached. Right panel: Add Input -> Your Datasets -> bdsl-k400.")
    here = pathlib.Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "pyproject.toml").is_file() and (c / "kaggle" / "src" / "k400").is_dir():
            return c
    raise FileNotFoundError("Open this notebook inside the project folder, or on Kaggle with the dataset attached.")

DATASET_ROOT = _find_root()
CODE_ROOT = DATASET_ROOT / "code" if (DATASET_ROOT / "code" / "pyproject.toml").is_file() else DATASET_ROOT
for _p in (CODE_ROOT / "kaggle" / "src", CODE_ROOT / "src"):
    sys.path.insert(0, str(_p))
from k400.setup import setup
S = setup(DATASET_ROOT, CODE_ROOT, NOTEBOOK)

In [ ]:
rc_report = S.k400("report")

In [ ]:
from IPython.display import Image, Markdown, display
import glob
for f in sorted(glob.glob(str(S.state_root / "outputs/figures/*.png"))):
    print(pathlib.Path(f).name); display(Image(filename=f))
display(Markdown((S.state_root / "outputs/RESULTS_SUMMARY.md").read_text(encoding="utf-8")[:6000]))

In [ ]:
S.finish(checks=[
    ("report built", rc_report == 0),
    ("final test results present", (S.state_root / "outputs/FINAL_TEST_LOCK.json").is_file()),
])